# 🧬 양자 인과 발견: Grover 알고리즘으로 단백질 신호전달 네트워크 탐색

Sachs(2005) 단백질 신호전달 데이터에서 MAPK 캐스케이드(Raf → Mek → Erk)의  
인과 구조를 **Grover 알고리즘**으로 탐색하고, 고전 전수조사와 비교합니다.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import networkx as nx

from src.data_loader import load_sachs_subset
from src.dag_utils import (
    get_variable_names, get_edge_list, enumerate_all_dags,
    get_ground_truth_dag, bitstring_to_dag, dag_to_bitstring,
    structural_hamming_distance, edge_metrics
)
from src.scoring import score_all_dags
from src.grover_search import run_grover_search

## 1. 데이터 로드 및 탐색

In [ ]:
variables = get_variable_names(3)  # Raf, Mek, Erk
data = load_sachs_subset(variables)
print(f'변수: {variables}')
print(f'샘플 수: {len(data):,}')
data.describe()

In [ ]:
# 상관관계
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 히트맵
corr = data.corr()
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(variables)
axes[0].set_yticks(range(3)); axes[0].set_yticklabels(variables)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=14)
axes[0].set_title('Pearson Correlation')
plt.colorbar(im, ax=axes[0])

# 정답 DAG
gt = get_ground_truth_dag(variables)
pos = {'Raf': (0, 0.5), 'Mek': (1, 1), 'Erk': (2, 0.5)}
nx.draw_networkx_nodes(gt, pos, ax=axes[1], node_color='#4ECDC4', node_size=1500, edgecolors='#2C3E50', linewidths=2)
nx.draw_networkx_labels(gt, pos, ax=axes[1], font_size=13, font_weight='bold')
nx.draw_networkx_edges(gt, pos, ax=axes[1], edge_color='#2C3E50', width=2.5, arrows=True, arrowsize=25, connectionstyle='arc3,rad=0.1')
axes[1].set_title('Ground Truth DAG (Sachs 2005)', fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. DAG 탐색 공간 정의

In [ ]:
edge_list = get_edge_list(variables)
valid_dags, _ = enumerate_all_dags(variables)
n_edges = len(edge_list)

print(f'가능한 방향 엣지: {n_edges}개')
for i, (src, dst) in enumerate(edge_list):
    print(f'  bit {i}: {src} → {dst}')
print(f'\n전체 후보: 2^{n_edges} = {2**n_edges}개')
print(f'유효 DAG (비순환): {len(valid_dags)}개')

## 3. 고전 전수조사: BDeu 점수로 모든 DAG 평가

In [ ]:
scored = score_all_dags(data, valid_dags, variables, equivalent_sample_size=10)

gt_bs = dag_to_bitstring(gt, edge_list)
print('BDeu 상위 10개 DAG:')
print(f'{"순위":>4} {"BDeu":>12} {"SHD":>4} {"엣지"}')
print('-' * 60)
for i, (bs, dag, sc) in enumerate(scored[:10]):
    shd = structural_hamming_distance(gt, dag)
    mark = ' ★' if bs == gt_bs else ''
    print(f'{i+1:>4} {sc:>12.2f} {shd:>4} {list(dag.edges())}{mark}')

for i, (bs, dag, sc) in enumerate(scored):
    if bs == gt_bs:
        print(f'\n정답 DAG 순위: {i+1}/{len(scored)}')

## 4. Grover 알고리즘으로 탐색

BDeu 상위 6개 DAG를 "좋은 구조"로 정의하고,  
Grover의 Oracle로 이들에 위상 반전을 적용합니다.

In [ ]:
top_k = 6
good_bitstrings = [scored[i][0] for i in range(top_k)]

print(f'타겟 DAG ({top_k}개):')
for bs in good_bitstrings:
    dag = bitstring_to_dag(bs, edge_list)
    print(f'  {bs} → {list(dag.edges())}')

N = 2 ** n_edges
M = len(good_bitstrings)
optimal_iters = int(np.pi / 4 * np.sqrt(N / M))
print(f'\n탐색 공간: N = {N}')
print(f'타겟 수: M = {M}')
print(f'최적 반복 횟수: π/4 × √(N/M) ≈ {optimal_iters}')

In [ ]:
# Grover 실행
grover_result = run_grover_search(
    n_qubits=n_edges,
    good_bitstrings=good_bitstrings,
    shots=4096
)

print(f'Grover 반복: {grover_result["n_iterations"]}회')
print(f'회로 깊이: {grover_result["circuit_depth"]}')
print(f'타겟 적중률: {grover_result["good_probability"]*100:.1f}%')
print(f'실행 시간: {grover_result["elapsed_time"]:.3f}초')

In [ ]:
# 측정 결과 히스토그램
fig, ax = plt.subplots(figsize=(12, 5))
sorted_counts = sorted(grover_result['counts'].items(), key=lambda x: x[1], reverse=True)[:20]
bs_list = [x[0] for x in sorted_counts]
val_list = [x[1] for x in sorted_counts]
colors = ['#E74C3C' if bs in good_bitstrings else '#BDC3C7' for bs in bs_list]

ax.bar(range(len(bs_list)), val_list, color=colors, edgecolor='#2C3E50')
ax.set_xticks(range(len(bs_list)))
ax.set_xticklabels(bs_list, rotation=90, fontsize=9)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Grover Measurement Results (red = target DAGs)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 양자 회로 시각화
grover_result['circuit'].draw(output='mpl', fold=30)

## 5. 고전 vs 양자 비교

In [ ]:
# 세 DAG 비교
best_bs, best_dag, best_sc = scored[0]
grover_dag = bitstring_to_dag(grover_result['top_bitstring'], edge_list)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pos = {'Raf': (0, 0.5), 'Mek': (1, 1), 'Erk': (2, 0.5)}

for ax, G, title in [
    (axes[0], gt, f'Ground Truth\nRaf → Mek → Erk'),
    (axes[1], best_dag, f'Classical Best\nSHD={structural_hamming_distance(gt, best_dag)}'),
    (axes[2], grover_dag, f'Grover Best\nSHD={structural_hamming_distance(gt, grover_dag)}')
]:
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='#4ECDC4', node_size=1500,
                           edgecolors='#2C3E50', linewidths=2)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=13, font_weight='bold')
    
    gt_edges = set(gt.edges())
    edge_colors = ['#27AE60' if e in gt_edges else '#E74C3C' for e in G.edges()]
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                           width=2.5, arrows=True, arrowsize=25,
                           connectionstyle='arc3,rad=0.1')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Ground Truth vs Classical vs Grover', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('초록 = 정답 엣지, 빨강 = 오답 엣지')

In [ ]:
# 정량 비교
cl_m = edge_metrics(gt, best_dag)
gr_m = edge_metrics(gt, grover_dag)

print(f'{"":>20} {"Classical":>15} {"Grover":>15}')
print('=' * 52)
print(f'{"평가 횟수":>20} {len(valid_dags):>15} {grover_result["n_iterations"]:>12} iter')
print(f'{"SHD":>20} {structural_hamming_distance(gt, best_dag):>15} {structural_hamming_distance(gt, grover_dag):>15}')
print(f'{"F1":>20} {cl_m["f1"]:>15.3f} {gr_m["f1"]:>15.3f}')
print(f'{"Precision":>20} {cl_m["precision"]:>15.3f} {gr_m["precision"]:>15.3f}')
print(f'{"Recall":>20} {cl_m["recall"]:>15.3f} {gr_m["recall"]:>15.3f}')
print(f'{"타겟 적중률":>20} {"100%":>15} {grover_result["good_probability"]*100:>14.1f}%')

## 6. 결론 및 회고

### 핵심 결과
1. **Grover의 이차 속도향상**: 64개 후보 중 최적 구조를 2회 반복만으로 100% 확률로 탐색
2. **마코프 동치류 한계**: 관측 데이터만으로는 동일 BDeu 점수의 DAG들을 구별 불가
3. **양자 우위의 조건**: 변수가 늘어나면 DAG 공간이 초지수적 증가 → Grover의 √N 이점 부각

### 양자적 접근의 의의
- 인과 발견을 **조합 최적화 문제**로 정식화하면 양자 알고리즘 적용 가능
- 현재 시뮬레이터 환경에서는 고전이 더 빠르지만, 변수 수 증가 시 양자 탐색의 이론적 우위 존재
- Oracle 구성의 효율성이 실제 양자 우위의 핵심 과제

### 한계 및 향후 과제
- Oracle에 BIC 점수를 직접 인코딩하는 방법 (현재는 사전 계산 필요)
- QAOA로 인과 구조 학습 (연속 최적화 접근)
- 개입(intervention) 데이터 활용으로 마코프 동치류 해소